**What is covered in `experimments_1` notebook?**
- Extracting Text page-wise, block-wise, even from multi-column magazine style layout in correct reading order
- Extracting images and display opr save them page wise
- Extracting images with in-image captions

**What am I going to cover in this notebook?**

- Distinguish "Body Text" from "headers" and "footers"
- Identifying "heading", "subheading", and "section breaks"
- Detect "Page numbers", "watermarks", and "foot notes"
- Preserve "line breaks" where they matter and ignore them where they do not
- Handle rotated text or sideways annotations
- Reconstruct reading order in documents with mixed layout styles:
    - separate images + captions
    - main content text blocks with correct reading orders (with heading sub-heading etc)
    - footer notes, page no, etc
    - compare it with "to_markdown" function.
- 

In [38]:
import fitz
from pprint import pprint
from pathlib import Path
from docling.document_converter import DocumentConverter


In [39]:
fitz_pdf = fitz.open(file_path)

In [14]:
data_folder_path = Path("/mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/iid-platform/sample_data/named")
file_path = data_folder_path / "nature-alberta-spring-2026-magazine-medium-mix-multi-col-text.pdf"
file_path

PosixPath('/mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/iid-platform/sample_data/named/nature-alberta-spring-2026-magazine-medium-mix-multi-col-text.pdf')

In [15]:
converter = DocumentConverter()
doc = converter.convert(file_path).document

[INFO] 2026-04-24 17:08:17,984 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-24 17:08:18,332 [RapidOCR] download_file.py:60: File exists and is valid: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-04-24 17:08:18,333 [RapidOCR] main.py:53: Using /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-04-24 17:08:18,844 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-24 17:08:18,919 [RapidOCR] download_file.py:60: File exists and is valid: /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-04-24 17:08:18,920 [RapidOCR] main.py:53: Using /mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/.llmenv/lib/python3.10/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer

In [16]:
# doc.export_to_markdown()

In [17]:
type(doc)

docling_core.types.doc.document.DoclingDocument

In [18]:
docdict = dict(doc)
docdict

{'schema_name': 'DoclingDocument',
 'version': '1.10.0',
 'name': 'nature-alberta-spring-2026-magazine-medium-mix-multi-col-text',
 'origin': DocumentOrigin(mimetype='application/pdf', binary_hash=9295684724258981057, filename='nature-alberta-spring-2026-magazine-medium-mix-multi-col-text.pdf', uri=None),
 'furniture': GroupItem(self_ref='#/furniture', parent=None, children=[], content_layer=<ContentLayer.FURNITURE: 'furniture'>, meta=None, name='_root_', label=<GroupLabel.UNSPECIFIED: 'unspecified'>),
 'body': GroupItem(self_ref='#/body', parent=None, children=[RefItem(cref='#/texts/0'), RefItem(cref='#/pictures/0'), RefItem(cref='#/texts/5'), RefItem(cref='#/texts/6'), RefItem(cref='#/texts/7'), RefItem(cref='#/texts/8'), RefItem(cref='#/texts/9'), RefItem(cref='#/texts/10'), RefItem(cref='#/texts/11'), RefItem(cref='#/pictures/1'), RefItem(cref='#/texts/14'), RefItem(cref='#/texts/15'), RefItem(cref='#/texts/16'), RefItem(cref='#/texts/17'), RefItem(cref='#/pictures/2'), RefItem(cre

In [21]:
docdict.keys()

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'field_regions', 'field_items', 'pages'])

In [23]:
myset = set()
for e in docdict['texts']:  # texts pictures
    myset.add(type(e))
myset

{docling_core.types.doc.document.ListItem,
 docling_core.types.doc.document.SectionHeaderItem,
 docling_core.types.doc.document.TextItem}

**Observation :**

**docling_core.types.doc.document.DoclingDocument**

* **'schema_name'**     : 'DoclingDocument'
* **'version'**         : '1.10.0'
* **'name'**            : 'file_name_excluding_extension'
* **'origin'**          : A `DocumentOrigin` object, which contains filetype (for ex pdf, etc) under mimetype
                          key or attribute, a 19 digit integer as 'bynary_hash'
* **'furniture'**       : A `GroupItem` object, which stores parts of a document that are structural or 
                          decorative, not part of the main content
* **'body'**            : A `GroupItem` object, which has 'self_ref' as "#/body", 'parent' as None, 
                          'children' as a list of reference items (`RefItem`s), the RefItem looks like:
                          RefItem(cref='#/<`texts` or `pictures` as str>/<index or serial num: int>')
* **'groups'**            A list of `ListGroup` objects or `GroupItem` objects or mix of both. Any item or
                          element in the `DoclingDocument['groups']` list irrespective of ListGroup or 
                          GroupItem it has 'self_ref' as `"#/groups/<incremental serial num: int>"`. 
                          'parent' may be RefItem(cref='#/body') or some other item with apt ref. 
                          'children' will be a list of reference items (`RefItem`s), 'children' maybe empty
                          list in some cases.  
* **'text'**            : A list of any objects out of the following: `SectionHeaderItem`, `TextItem`, 
                          or `ListItem`.
                          Any item or element in the `DoclingDocument['texts']` list irrespective of 
                          SectionHeaderItem or TextItem or ListItem it has 'self_ref' as 
                          `"#/texts/<incremental serial num: int>"`.
                          Any of these elements may have a list of reference items (`RefItem`s) as 'children'.
                          Elements inside `DoclingDocument['texts']` have an extra attribute `prov`, 
                          which contains a list of `ProvenanceItem` objects.
                          `ProvenanceItem` contains page number (page_no), bounding box (bbox), coordinate
                          origin (coord_origin), character span (charspan), the text within this block (orig
                          or text attribute), it may contain hyperlink. In case of SectionHeaderItem it 
                          contains the section header level as integer. 
* **'pictures'**        : Contains a list of `PictureItem`s. A PictureItem element has 'self_ref' as 
                          `"#/pictures/<incremental serial num: int>"`, 'children' as a list of reference items
                          for text blocks, these text blocks are in-image captions, in 'prov' key it contains 
                          a list of `ProvenanceItem` objects and bounding box (bbox) information inside it, 
                          this bbox is for the image, the cropped portion of the pdf. 
* **'tables'**          : Empty list if no tables detected in the pdf or document. 
* **'key_value_items'** : Empty list if no key value items detected in the pdf or document.
* **'form_items'**      : Empty list if no form items detected in the pdf or document.
* **'pages'**           : A dictionary, keys are indices 1,2,3,...
                          Each index stores a PageItem containing "Size" with height and width of that page, 
                          It also stores the page number (page_no: int) of this page item as per the pdf or doc.

**What to do next?**
1. **Understand what are 'groups':** Check different group items like `ListGroup` or `GroupItem`, match with original pdf
2. **Understand different kinds of 'texts':** Check different text items like `SectionHeaderItem` or `TextItem` or `ListItem`, match with original pdf
3. Check `PictureItem`s and the ref text blocks, and match with original pdf

**Understand what are 'groups'**
- Let's examine the first group (idx 0) from the list

In [27]:
group0 = docdict['groups'][0] 
pprint(dict(group0))

{'children': [RefItem(cref='#/texts/24'),
              RefItem(cref='#/texts/25'),
              RefItem(cref='#/texts/26'),
              RefItem(cref='#/texts/27'),
              RefItem(cref='#/texts/28'),
              RefItem(cref='#/texts/29'),
              RefItem(cref='#/texts/30'),
              RefItem(cref='#/texts/31'),
              RefItem(cref='#/texts/32'),
              RefItem(cref='#/texts/33'),
              RefItem(cref='#/texts/34'),
              RefItem(cref='#/texts/35'),
              RefItem(cref='#/texts/36'),
              RefItem(cref='#/texts/37')],
 'content_layer': <ContentLayer.BODY: 'body'>,
 'label': <GroupLabel.LIST: 'list'>,
 'meta': None,
 'name': 'list',
 'parent': RefItem(cref='#/body'),
 'self_ref': '#/groups/0'}


In [28]:
# Let's print the text content from each of these `RefItem`s and also in the original pdf 
# to explore why these blocks are kept in the same group.

In [42]:
# list of integers of the text blocks for group with index 0
blocksidx = list(range(24, 38))
# make the 'texts' list to dict with the text block idx as key
textblocks = {}  # key=block_index ; value=tuple(page_no, text_content)
for ele in docdict['texts']:
    blockidx = int(list(ele.self_ref.split('/'))[-1])
    # textblocks dict :: key=block_index ; value=tuple(page_no, text_content)
    textblocks[blockidx] = tuple([ele.prov[0].page_no, ele.text])
# Print the targeted blocks under group 0
print("Printing for group 0 : This is a ListGroup object")
print(f"Involved block indices are: {blocksidx}\n")
for idx in blocksidx:
    print(f"Idx: {idx}  |  PageNo: {textblocks[idx][0]}  |  Text: {textblocks[idx][1]}")

Printing for group 0 : This is a ListGroup object
Involved block indices are: [24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]

Idx: 24  |  PageNo: 3  |  Text: 2 President's Message
Idx: 25  |  PageNo: 3  |  Text: 3 Nature Alberta News
Idx: 26  |  PageNo: 3  |  Text: 4 Caribou Update: Plans for Extirpation
Idx: 27  |  PageNo: 3  |  Text: 6 A Hummingbird Primer
Idx: 28  |  PageNo: 3  |  Text: 11 Of Tree Rings and Tea Leaves
Idx: 29  |  PageNo: 3  |  Text: 14 Botanical Treasures of Hinton's Beaver Boardwalk
Idx: 30  |  PageNo: 3  |  Text: 18 Coal Confusion: Throwing Dust in the Air
Idx: 31  |  PageNo: 3  |  Text: 23 Book Review: Frank Farley and the Birds of Alberta
Idx: 32  |  PageNo: 3  |  Text: 24 Sharing Space: For the Love of Honkers
Idx: 33  |  PageNo: 3  |  Text: 25 Your Shot
Idx: 34  |  PageNo: 3  |  Text: 26 Disappearing Peace: The Mountain Meadows of Alberta's Rockies
Idx: 35  |  PageNo: 3  |  Text: 32 Spring Spawn: Northern Pike and Walleye
Idx: 36  |  PageNo: 3  |  Te

**Comment:**

- In the original pdf doc this is the contents, which is a group of text elements
- This was a ListGroup object, let's check the next group elements (idx 1 and 2), which are GroupItem objects.
- There is no children in groups index 1, so only idx 2 will be examined

In [41]:
group1 = docdict['groups'][1]
group2 = docdict['groups'][2]
pprint(dict(group1))
print()
pprint(dict(group2))

{'children': [],
 'content_layer': <ContentLayer.BODY: 'body'>,
 'label': <GroupLabel.KEY_VALUE_AREA: 'key_value_area'>,
 'meta': None,
 'name': 'group',
 'parent': RefItem(cref='#/body'),
 'self_ref': '#/groups/1'}

{'children': [RefItem(cref='#/texts/47'),
              RefItem(cref='#/texts/48'),
              RefItem(cref='#/texts/49'),
              RefItem(cref='#/texts/50'),
              RefItem(cref='#/texts/51'),
              RefItem(cref='#/texts/52'),
              RefItem(cref='#/texts/53'),
              RefItem(cref='#/texts/54'),
              RefItem(cref='#/texts/55'),
              RefItem(cref='#/texts/56'),
              RefItem(cref='#/texts/57')],
 'content_layer': <ContentLayer.BODY: 'body'>,
 'label': <GroupLabel.KEY_VALUE_AREA: 'key_value_area'>,
 'meta': None,
 'name': 'group',
 'parent': RefItem(cref='#/body'),
 'self_ref': '#/groups/2'}


In [43]:
# list of integers of the text blocks for group with index 2
blocksidx = list(range(47, 58))
# make the 'texts' list to dict with the text block idx as key
textblocks = {}  # key=block_index ; value=tuple(page_no, text_content)
for ele in docdict['texts']:
    blockidx = int(list(ele.self_ref.split('/'))[-1])
    # textblocks dict :: key=block_index ; value=tuple(page_no, text_content)
    textblocks[blockidx] = tuple([ele.prov[0].page_no, ele.text])
# Print the targeted blocks under group 0
print("Printing for group idx 2 : This is a GroupItem object")
print(f"Involved block indices are: {blocksidx}\n")
for idx in blocksidx:
    print(f"Idx: {idx}  |  PageNo: {textblocks[idx][0]}  |  Text: {textblocks[idx][1]}")

Printing for group idx 2 : This is a GroupItem object
Involved block indices are: [47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57]

Idx: 47  |  PageNo: 4  |  Text: Publisher Nature Alberta
Idx: 48  |  PageNo: 4  |  Text: Executive Director
Idx: 49  |  PageNo: 4  |  Text: Stephanie Weizenbach
Idx: 50  |  PageNo: 4  |  Text: Editor-in-Chief Erin McCloskey
Idx: 51  |  PageNo: 4  |  Text: Managing Editor Jason Switner
Idx: 52  |  PageNo: 4  |  Text: Creative Susan May, intrinsic design
Idx: 53  |  PageNo: 4  |  Text: Cover Image Myrna Pearman
Idx: 54  |  PageNo: 4  |  Text: Nature Alberta Magazine is published four times per year by:
Idx: 55  |  PageNo: 4  |  Text: Nature Alberta 11759 Groat Road Edmonton, AB  T5M 3K6 (780) 427-8124
Idx: 56  |  PageNo: 4  |  Text: info@naturealberta.ca
Idx: 57  |  PageNo: 4  |  Text: Content editorinchief@naturealberta.ca


**Comment:**
- This was another group of text blocks from page 4

**Understand different kinds of 'texts'**
- Few `SectionHeaderItem`s are : [ `self_ref='#/texts/0'`, `self_ref='#/texts/70'` ]
- Few `TextItem`s are : [ `self_ref='#/texts/1'`, `self_ref='#/texts/60'`, `self_ref='#/texts/68'` ]
- Few `ListItem`s are : [ `self_ref='#/texts/214'`, `self_ref='#/texts/215'`, `self_ref='#/texts/216'` ]

In [44]:
# SectionHeaderItem
indices = [0, 70]
for idx in indices:
    print(f"Idx: {idx}  |  PageNo: {textblocks[idx][0]}  |  Text: {textblocks[idx][1]}")

Idx: 0  |  PageNo: 1  |  Text: NATURE ALBERTA
Idx: 70  |  PageNo: 4  |  Text: PRESIDENT'S MESSAGE


In [45]:
# TextItem
indices = [1, 60, 68]
for idx in indices:
    print(f"Idx: {idx}  |  PageNo: {textblocks[idx][0]}  |  Text: {textblocks[idx][1]}")

Idx: 1  |  PageNo: 1  |  Text: SPRING 2026
Idx: 60  |  PageNo: 4  |  Text: Nature Alberta Magazine (electronic) is made available free of charge at naturealberta.ca . Print copies of Nature Alberta Magazine are available by annual subscription for $32 (Canadian funds + GST), which covers four issues per year, plus postage and handling, within Canada. Subscriptions can be purchased at naturealberta.ca/store . Publications Mail Agreement No. 40015475
Idx: 68  |  PageNo: 4  |  Text: LOVE OF NATURE


In [47]:
# ListItem
indices = [214, 215, 216]
for idx in indices:
    print(f"Idx: {idx}  |  PageNo: {textblocks[idx][0]}  |  Text: {textblocks[idx][1]}")

Idx: 214  |  PageNo: 7  |  Text:  Environment and Climate Change Canada (2020). Agreement for the conservation and recovery of the Woodland Caribou in Alberta. canada. ca/en/environment-climate-change/ services/species-risk-public-registry/ conservation-agreements/agreementconservation-woodland-caribouboreal-alberta-2020.html
Idx: 215  |  PageNo: 7  |  Text:  Environment Canada (2012). Recovery Strategy for the Woodland Caribou (Rangifer tarandus caribou), Boreal population, in Canada. publications.gc.ca/ collections/ collection_2012/ ec/En3-4140-2012-eng.pdf
Idx: 216  |  PageNo: 7  |  Text:  Government of Alberta (2011). A Woodland Caribou Policy for Alberta. open.alberta.ca/publications/ 9780778594789


**Comments:**
- `SectionHeaderItem`s are just normal headings/subheadings
- `TextItem`s are normal text blocks without any other identity
- But `ListItem`s are something special. The ListItems with text block indices 214, 215, and 216 are actual list item elements under "References" at page 7, each text block is a list item, and total of 3 references given at page 7. In these elements parent is given as "#/groups/7", so these kind of things are stored at text block level under `'texts'` and also at group level under `'

- section header with different levels
- where are footers and etc
- 